# ML-04: Search Intelligence Data Contract

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanraza04/flyrank_intern_content/blob/main/work/notebooks/w03_data_contract.ipynb)

This notebook documents the data contract for my **Refresh / Content Opportunity Scoring** lane. It uses a mid-panel month, March 2026. I deliberately do not use the June sample table because June is the sealed final outcome month.

## Before running in Colab

1. Open this notebook with the Colab badge above.
2. In Colab, open the left sidebar's **Secrets** panel (key icon).
3. Add a secret named `HF_TOKEN`, paste your Hugging Face **Read** token as its value, and enable notebook access.
4. Run all cells. The token is read from Colab Secrets and is never written into this notebook or committed to GitHub.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError(
        'Set HF_TOKEN in Colab Secrets (or in your local environment). ' 
        'Do not paste a token into this notebook.'
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('Connection ready. The notebook will query the March 2026 partition only.')

## 1. My lane's data contract

**Lane:** Refresh / Content Opportunity Scoring.

**What one row means:** In the source table, one row is one daily Search Console observation for one `client_hash_id`, one `content_hash_id`, and one `report_date`. In the modeling frame later in this notebook, one row becomes one client-content item observed at the March 14 decision point.

**Table used:** `fact_content_daily_performance`, queried only from its March 2026 partition. This table supplies daily impressions, clicks, average position, and the GA4 availability flag.

**Time window:** March 1 to March 14 is the feature window. March 15 to March 31 is a later, within-month proxy outcome window. March is mid-panel, so it is appropriate for development.

**What I would predict or rank:** I would rank pages likely to experience a meaningful drop in search impressions over the next 17 days. For this exercise, `is_declining_proxy` is 1 when a page with at least 100 first-window impressions receives less than 80% of its expected impressions in the later window.

**What I deliberately exclude:** I exclude GA4 engagement features from this first model. The warehouse records zeros before a client's GA4 connection starts, and `ga4_data_available` is not true for every daily row. I keep this first slice consistently based on GSC data, while still measuring GA4 availability below.

## 2. Three verification queries

These are the only three verification queries in this notebook. They check the grain, the March slice size and dates, and the availability rule.

### Verification query 1: Does the source grain match the contract?

If `duplicate_grain_keys` is zero, `report_date + client_hash_id + content_hash_id` uniquely identifies every March source row.

In [ ]:
grain_check = con.sql(f"""
WITH keyed AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM {MARCH_DAILY}
    GROUP BY 1, 2, 3
)
SELECT
    SUM(rows_per_key) AS source_rows,
    COUNT(*) AS distinct_grain_keys,
    COUNT(*) FILTER (WHERE rows_per_key > 1) AS duplicate_grain_keys
FROM keyed
""").df()
grain_check

### Verification query 2: How large is the March slice, and what dates does it cover?

In [ ]:
slice_check = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {MARCH_DAILY}
""").df()
slice_check

### Verification query 3: How much of the slice has GA4 data available?

The `IS TRUE` condition matters. It retains only rows where GA4 is genuinely available, rather than treating unavailable or missing data as usable zeros.

In [ ]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*),
        2
    ) AS ga4_available_pct
FROM {MARCH_DAILY}
""").df()
availability_check

## 3. Five honest features and the target proxy

The query below produces a real page-level modeling frame. One row represents one client-content item as it was known on March 14. The later March window is used only to create the proxy label.

1. `impressions_first14`: knowable at the decision moment because Search Console has already recorded impressions from March 1 to March 14.
2. `clicks_first14`: knowable at the decision moment because Search Console has already recorded clicks from March 1 to March 14.
3. `avg_position_first14`: knowable at the decision moment because it is an average of positions observed before March 15.
4. `active_search_days_first14`: knowable at the decision moment because it counts days with observed impressions in the completed first window.
5. `ctr_first14`: knowable at the decision moment because it is calculated only from first-window clicks and impressions.

In [ ]:
feature_frame = con.sql(f"""
WITH first_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_first14,
        SUM(gsc_clicks) AS clicks_first14,
        AVG(gsc_avg_position) AS avg_position_first14,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_search_days_first14
    FROM {MARCH_DAILY}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
    GROUP BY 1, 2
),
later_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next17
    FROM {MARCH_DAILY}
    WHERE report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-31'
    GROUP BY 1, 2
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_first14,
    f.clicks_first14,
    f.avg_position_first14,
    f.active_search_days_first14,
    100.0 * f.clicks_first14 / NULLIF(f.impressions_first14, 0) AS ctr_first14,
    l.impressions_next17,
    CASE
        WHEN f.impressions_first14 >= 100
         AND COALESCE(l.impressions_next17, 0) < 0.80 * f.impressions_first14 * 17.0 / 14.0
        THEN 1 ELSE 0
    END AS is_declining_proxy
FROM first_window AS f
LEFT JOIN later_window AS l USING (client_hash_id, content_hash_id)
WHERE f.impressions_first14 >= 100
""").df()

print(f'One row = one client-content item at the March 14 decision point: {len(feature_frame):,} rows')
print(f"Declining-proxy rate: {feature_frame['is_declining_proxy'].mean():.1%}")
feature_frame.head()

## 4. The leakage trap

I use a small decision tree only to demonstrate leakage, not to claim that it is a finished model. The honest version can use only the five first-window features. Then I deliberately add `leaky_later_impression_ratio`, which is calculated from the future outcome window and therefore contains label information. Its score should become suspiciously high. I delete the leaky column immediately afterwards and retain the honest score.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

honest_features = [
    'impressions_first14',
    'clicks_first14',
    'avg_position_first14',
    'active_search_days_first14',
    'ctr_first14',
]

model_frame = feature_frame.dropna(subset=honest_features).copy()
if len(model_frame) > 100_000:
    model_frame = model_frame.sample(100_000, random_state=42)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(
    splitter.split(model_frame, groups=model_frame['client_hash_id'])
)

def quick_score(columns):
    model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=42)
    model.fit(
        model_frame.iloc[train_idx][columns],
        model_frame.iloc[train_idx]['is_declining_proxy'],
    )
    probability = model.predict_proba(model_frame.iloc[test_idx][columns])[:, 1]
    actual = model_frame.iloc[test_idx]['is_declining_proxy'].to_numpy()
    top_20 = np.argsort(probability)[-20:]
    return {
        'roc_auc': round(roc_auc_score(actual, probability), 3),
        'precision_at_20': round(precision_score(actual[top_20], np.ones(len(top_20))), 3),
    }

honest_score = quick_score(honest_features)
honest_score

In [ ]:
# Deliberate leakage: this uses impressions from March 15 to March 31.
model_frame['leaky_later_impression_ratio'] = (
    model_frame['impressions_next17']
    / (model_frame['impressions_first14'] * 17.0 / 14.0)
)

leaky_score = quick_score(honest_features + ['leaky_later_impression_ratio'])
print({'honest_score': honest_score, 'leaky_score': leaky_score})

# Remove the future-derived feature. It must not be part of the real feature set.
del model_frame['leaky_later_impression_ratio']
assert 'leaky_later_impression_ratio' not in model_frame.columns
print(f'Honest score retained: {honest_score}')

## 5. Limitation and self-check

**Named limitation:** This is a short, within-March proxy rather than a validated editorial outcome. A fall in impressions can come from seasonality, demand changes, tracking changes, or a real content problem. It does not prove that refreshing the page will improve performance.

- [x] I named the row grain, table, time window, target proxy, and one exclusion.
- [x] I wrote exactly three verification queries, including availability filtered with `IS TRUE`.
- [x] I built a five-feature frame and explained when each feature is available.
- [x] I demonstrated a label-derived leakage feature, removed it, and kept the honest score.
- [x] I kept the final June sample sealed and used March 2026 for development.